# Notebook 02 — Preprocesamiento y Pipeline Experimental
**TFM:** Detección de fraude en tarjetas de crédito — comparativa de modelos y generalización cross-domain

**Prerrequisito:** Haber ejecutado `notebook_01_EDA.ipynb` y tener `creditcard.csv` en la misma carpeta.

**Librerías necesarias:**
```
pip install pandas numpy scikit-learn imbalanced-learn xgboost lightgbm joblib
```


In [1]:
# ── INSTALACIÓN (ejecutar solo si es necesario) ──────────────────────────
!pip install pandas numpy scikit-learn imbalanced-learn xgboost lightgbm joblib


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── IMPORTACIONES ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, time, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_style('whitegrid')
np.random.seed(42)

print("✅ Librerías cargadas correctamente")


✅ Librerías cargadas correctamente


## 1. Carga y preprocesamiento inicial

In [3]:
# ── CARGA ─────────────────────────────────────────────────────────────────
df = pd.read_csv('creditcard.csv')
print(f"Dataset cargado: {df.shape[0]:,} transacciones × {df.shape[1]} variables")
print(f"Fraudes: {df['Class'].sum():,} ({df['Class'].mean()*100:.3f}%)")


Dataset cargado: 284,807 transacciones × 31 variables
Fraudes: 492 (0.173%)


In [4]:
# ── SEPARAR FEATURES Y TARGET ─────────────────────────────────────────────
X = df.drop('Class', axis=1).copy()
y = df['Class'].copy()

# Escalar Time y Amount (las únicas variables no-PCA)
scaler_time   = StandardScaler()
scaler_amount = StandardScaler()

X['Time']   = scaler_time.fit_transform(X[['Time']])
X['Amount'] = scaler_amount.fit_transform(X[['Amount']])

print("Variables escaladas: Time y Amount")
print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")


Variables escaladas: Time y Amount
Shape X: (284807, 30)
Shape y: (284807,)


## 2. División estratificada entrenamiento / test

In [5]:
# ── DIVISIÓN 80/20 ESTRATIFICADA ──────────────────────────────────────────
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("=== PARTICIÓN ENTRENAMIENTO / TEST ===")
print(f"  Entrenamiento : {X_train.shape[0]:>8,} muestras")
print(f"    Fraudes     : {y_train.sum():>8,} ({y_train.mean()*100:.3f}%)")
print(f"  Test          : {X_test.shape[0]:>8,} muestras")
print(f"    Fraudes     : {y_test.sum():>8,} ({y_test.mean()*100:.3f}%)")
print()
print("✅ Proporción de fraude conservada en ambas particiones (split estratificado)")


=== PARTICIÓN ENTRENAMIENTO / TEST ===
  Entrenamiento :  227,845 muestras
    Fraudes     :      394 (0.173%)
  Test          :   56,962 muestras
    Fraudes     :       98 (0.172%)

✅ Proporción de fraude conservada en ambas particiones (split estratificado)


In [6]:
# ── GUARDAR PARTICIONES PARA USO EN OTROS NOTEBOOKS ──────────────────────
os.makedirs('data_splits', exist_ok=True)

# Resetear índice antes de guardar para evitar errores con pyarrow
X_train_save = X_train.reset_index(drop=True)
X_test_save  = X_test.reset_index(drop=True)
y_train_save = y_train.reset_index(drop=True)
y_test_save  = y_test.reset_index(drop=True)

X_train_save.to_csv('data_splits/X_train.csv', index=False)
X_test_save.to_csv('data_splits/X_test.csv', index=False)
y_train_save.to_csv('data_splits/y_train.csv', index=False)
y_test_save.to_csv('data_splits/y_test.csv', index=False)

print("✅ Particiones guardadas en data_splits/")
print("   X_train.csv | X_test.csv | y_train.csv | y_test.csv")

✅ Particiones guardadas en data_splits/
   X_train.csv | X_test.csv | y_train.csv | y_test.csv


## 3. Definición del protocolo experimental

In [7]:
# ── CONFIGURACIÓN EXPERIMENTAL ────────────────────────────────────────────
# Validación cruzada
CV_FOLDS    = 5
RANDOM_SEED = 42
SCORING_METRIC = 'average_precision'  # AUC-PR

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

print("=== PROTOCOLO EXPERIMENTAL ===")
print(f"  Folds de validación cruzada : {CV_FOLDS}")
print(f"  Métrica principal (scoring) : AUC-PR (average_precision)")
print(f"  Semilla aleatoria           : {RANDOM_SEED}")
print(f"  Estrategia de partición     : StratifiedKFold (resampling DENTRO del fold)")


=== PROTOCOLO EXPERIMENTAL ===
  Folds de validación cruzada : 5
  Métrica principal (scoring) : AUC-PR (average_precision)
  Semilla aleatoria           : 42
  Estrategia de partición     : StratifiedKFold (resampling DENTRO del fold)


In [8]:
# ── DEFINICIÓN DE MODELOS BASELINE ────────────────────────────────────────
ratio_neg_pos = int((y_train == 0).sum() / (y_train == 1).sum())

MODELS = {
    'LR': LogisticRegression(
        solver='lbfgs', max_iter=1000,
        class_weight='balanced', random_state=RANDOM_SEED
    ),
    'DT': DecisionTreeClassifier(
        criterion='gini', class_weight='balanced',
        random_state=RANDOM_SEED
    ),
    'RF': RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    'XGB': XGBClassifier(
        tree_method='hist', scale_pos_weight=ratio_neg_pos,
        use_label_encoder=False, eval_metric='aucpr',
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    'LGBM': LGBMClassifier(
        is_unbalance=True, n_jobs=-1,
        random_state=RANDOM_SEED, verbose=-1
    )
}

print("=== MODELOS DEFINIDOS ===")
for name, model in MODELS.items():
    print(f"  {name:<5} → {model.__class__.__name__}")


=== MODELOS DEFINIDOS ===
  LR    → LogisticRegression
  DT    → DecisionTreeClassifier
  RF    → RandomForestClassifier
  XGB   → XGBClassifier
  LGBM  → LGBMClassifier


In [9]:
# ── DEFINICIÓN DE ESTRATEGIAS DE RESAMPLING ──────────────────────────────
RESAMPLERS = {
    'None' : None,
    'SMOTE': SMOTE(random_state=RANDOM_SEED, k_neighbors=5),
    'ADASYN': ADASYN(random_state=RANDOM_SEED, n_neighbors=5)
}

print("=== ESTRATEGIAS DE RESAMPLING ===")
for name, res in RESAMPLERS.items():
    print(f"  {name:<7} → {res.__class__.__name__ if res else 'Sin resampling'}")

print()
n_combos = len(MODELS) * len(RESAMPLERS)
print(f"Total combinaciones a evaluar: {len(MODELS)} modelos × {len(RESAMPLERS)} estrategias = {n_combos} combinaciones")


=== ESTRATEGIAS DE RESAMPLING ===
  None    → Sin resampling
  SMOTE   → SMOTE
  ADASYN  → ADASYN

Total combinaciones a evaluar: 5 modelos × 3 estrategias = 15 combinaciones


## 4. Función de construcción del pipeline

In [10]:
# ── FUNCIÓN PARA CONSTRUIR PIPELINES ──────────────────────────────────────
def build_pipeline(model, resampler=None):
    """
    Construye un pipeline que encadena (opcionalmente) el resampler
    con el clasificador. El resampling se aplica solo al conjunto de
    entrenamiento de cada fold, nunca al de validación.
    
    Parámetros:
        model     : clasificador sklearn/xgboost/lightgbm
        resampler : instancia de SMOTE, ADASYN, o None
    
    Retorna:
        pipeline  : ImbPipeline (imblearn) o sklearn Pipeline
    """
    if resampler is not None:
        steps = [('resampler', resampler), ('classifier', model)]
        return ImbPipeline(steps=steps)
    else:
        return Pipeline(steps=[('classifier', model)])

# Verificación
test_pipe = build_pipeline(MODELS['XGB'], RESAMPLERS['SMOTE'])
print("✅ Pipeline de prueba construido:")
print(f"   Pasos: {[step[0] for step in test_pipe.steps]}")


✅ Pipeline de prueba construido:
   Pasos: ['resampler', 'classifier']


## 5. Verificación del protocolo anti-leakage

In [11]:
# ── VERIFICACIÓN ANTI-DATA-LEAKAGE ────────────────────────────────────────
print("=== VERIFICACIÓN DEL PROTOCOLO ANTI-LEAKAGE ===")
print()
print("Fold | Train samples | Train fraudes | Val samples | Val fraudes | % fraude train | % fraude val")
print("-" * 95)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_fold_tr = X_train.iloc[tr_idx]
    y_fold_tr = y_train.iloc[tr_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Simular resampling solo en entrenamiento
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_fold_tr, y_fold_tr)
    
    print(f"  {fold}  | {len(X_fold_tr):>13,} | {y_fold_tr.sum():>13,} | {len(X_fold_val):>11,} | "
          f"{y_fold_val.sum():>11,} | {y_fold_tr.mean()*100:>14.3f}% | {y_fold_val.mean()*100:>12.3f}%")
    print(f"       | Tras SMOTE: {len(X_res):>10,} muestras (resampling NO aplicado a validación)")

print()
print("✅ SMOTE aplicado únicamente al entrenamiento de cada fold")
print("✅ El conjunto de validación NO está contaminado por instancias sintéticas")
print("✅ El conjunto de test (56,961 muestras) NO se ha tocado")


=== VERIFICACIÓN DEL PROTOCOLO ANTI-LEAKAGE ===

Fold | Train samples | Train fraudes | Val samples | Val fraudes | % fraude train | % fraude val
-----------------------------------------------------------------------------------------------
  1  |       182,276 |           316 |      45,569 |          78 |          0.173% |        0.171%
       | Tras SMOTE:    363,920 muestras (resampling NO aplicado a validación)
  2  |       182,276 |           315 |      45,569 |          79 |          0.173% |        0.173%
       | Tras SMOTE:    363,922 muestras (resampling NO aplicado a validación)
  3  |       182,276 |           315 |      45,569 |          79 |          0.173% |        0.173%
       | Tras SMOTE:    363,922 muestras (resampling NO aplicado a validación)
  4  |       182,276 |           315 |      45,569 |          79 |          0.173% |        0.173%
       | Tras SMOTE:    363,922 muestras (resampling NO aplicado a validación)
  5  |       182,276 |           315 |      45

## 6. Definición de métricas de evaluación

In [12]:
# ── MÉTRICAS DE EVALUACIÓN ────────────────────────────────────────────────
from sklearn.metrics import (
    average_precision_score, recall_score, f1_score,
    roc_auc_score, precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)

def evaluate_model(model, X_eval, y_eval, model_name='', strategy='', t_train=0, t_infer=0):
    """
    Calcula todas las métricas de evaluación para un modelo dado.
    
    Retorna dict con todas las métricas.
    """
    y_pred  = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]
    
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()
    
    return {
        'Modelo'           : model_name,
        'Estrategia'       : strategy,
        'AUC_PR'           : round(average_precision_score(y_eval, y_proba), 4),
        'AUC_ROC'          : round(roc_auc_score(y_eval, y_proba), 4),
        'Recall'           : round(recall_score(y_eval, y_pred), 4),
        'Precision'        : round(precision_score(y_eval, y_pred, zero_division=0), 4),
        'F1'               : round(f1_score(y_eval, y_pred), 4),
        'TP'               : int(tp),
        'FP'               : int(fp),
        'TN'               : int(tn),
        'FN'               : int(fn),
        'T_train_s'        : round(t_train, 3),
        'T_infer_ms_tx'    : round(t_infer * 1000 / len(X_eval), 4),
    }

print("✅ Función evaluate_model definida")
print("   Métricas: AUC_PR | AUC_ROC | Recall | Precision | F1 | TP | FP | TN | FN | T_train | T_infer")


✅ Función evaluate_model definida
   Métricas: AUC_PR | AUC_ROC | Recall | Precision | F1 | TP | FP | TN | FN | T_train | T_infer


In [13]:
# ── GUARDAR CONFIGURACIÓN PARA OTROS NOTEBOOKS ────────────────────────────
config = {
    'CV_FOLDS'      : CV_FOLDS,
    'RANDOM_SEED'   : RANDOM_SEED,
    'SCORING_METRIC': SCORING_METRIC,
    'ratio_neg_pos' : ratio_neg_pos,
    'n_train'       : len(X_train),
    'n_test'        : len(X_test),
    'n_fraud_train' : int(y_train.sum()),
    'n_fraud_test'  : int(y_test.sum()),
}

import json
with open('data_splits/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuración guardada en data_splits/config.json")
print()
print("=== RESUMEN FINAL DEL NOTEBOOK 02 ===")
print(f"  Particiones guardadas    : data_splits/")
print(f"  Combinaciones a evaluar  : {n_combos}")
print(f"  Configuración guardada   : data_splits/config.json")
print()
print("➡️  Siguiente paso: ejecutar notebook_03_modelos_ULB.ipynb")


✅ Configuración guardada en data_splits/config.json

=== RESUMEN FINAL DEL NOTEBOOK 02 ===
  Particiones guardadas    : data_splits/
  Combinaciones a evaluar  : 15
  Configuración guardada   : data_splits/config.json

➡️  Siguiente paso: ejecutar notebook_03_modelos_ULB.ipynb
